# Model 3: Llama-3.1-8B-Instruct, fine-tuned with QLoRA

**The question:** does a large language model (8B parameters, 64× the strong baseline), fine-tuned on our training set, beat the smaller trained models? It also gives the fine-tuned side of the final milestone's comparison with *prompted* LLMs.

**`meta-llama/Llama-3.1-8B-Instruct`** gets a chat prompt with the task description, the post and the article, and answers with the type and the spoiler:
```
type: multi
spoiler: bacon
spoiler: eggs
```
As with Flan-T5, one model does both subtasks.

**QLoRA** (Dettmers et al., 2023) makes an 8B model trainable on one 24 GB GPU:
- The base model is loaded in 4-bit NF4 (about 5.5 GB) and stays frozen.
- Small low-rank adapters are trained on all linear layers: rank 16, alpha 32, dropout 0.05, about 42M trainable parameters (0.5%).
- Gradient checkpointing and a paged 8-bit AdamW keep memory low.

The loss counts only the answer tokens, not the prompt. The article is cut to 1,600 tokens and the answer to 256, so a prompt plus answer stays under about 2,048 tokens. The cut hides every spoiler span of under 1% of training posts, which are left out of training.

**Learning-rate sweep:** 5e-5, 1e-4, 2e-4. Batch 1 with 16 accumulation steps (effective 16), 2 epochs, cosine schedule with 3% warmup, bf16. The best epoch is chosen by end-to-end dev BLEU with greedy decoding.

**Before the first run (once):**
1. Accept the Llama 3.1 license at https://huggingface.co/meta-llama/Llama-3.1-8B-Instruct (approval usually takes minutes to hours).
2. On the VM: `hf auth login` and paste a Hugging Face access token with read access (or `export HF_TOKEN=...`).

The model download is about 16 GB.

**Running it on the L4 VM** (about 12.5 hours: each learning rate takes about 4 hours, including dev generation after each epoch):
```bash
cd ~/project/milestone3
source ~/project/.venv/bin/activate
pip install -r requirements.txt               # once for all Milestone 3 notebooks
cp -r ../milestone2/tools .                   # the official METEOR jar, already downloaded in Milestone 2
tmux new -s m3                                # detach with Ctrl-b d, reattach with: tmux attach -t m3
jupyter nbconvert --to notebook --execute --inplace --ExecutePreprocessor.timeout=-1 3-llama-qlora.ipynb
```
`bash run-all.sh` runs all Milestone 3 notebooks one after the other instead. Checkpoints include the optimizer state, so an interrupted run resumes from its last epoch, and a learning rate that has finished (its `runs/.../run_info.json` exists) is skipped when the notebook is run again, so an interrupted sweep picks up where it stopped.

**Smoke test** (a few minutes on a laptop CPU; on the VM it uses the GPU): `SMOKE=1 jupyter nbconvert --to notebook --execute 3-llama-qlora.ipynb --output smoke-run.ipynb`. It swaps in a tiny model, trains 2 steps on 16 posts and writes everything under `smoke/`.

**Outputs:** adapters in `runs/llama-qlora-lr*/` (small, about 170 MB each; not committed); `output/llama-qlora-sweep.json`, predictions `output/llama-qlora-{dev,test}.jsonl` and `output/llama-qlora-goldtypes-test.jsonl`, and scores in `output/scores/`.

## Setup

Imports and run settings.
- **4-bit loading** needs `bitsandbytes`, which only works on CUDA. So it's used whenever a GPU is present, including the smoke test on the VM.
- `SMOKE=1` swaps in `HuggingFaceTB/SmolLM2-135M-Instruct`, a tiny chat model with the same kind of chat template, and uses shorter articles.
- On a laptop the smoke test runs on the CPU without 4-bit.

In [3]:
import gc
import json
import os
import subprocess
import sys
from pathlib import Path

import pandas as pd
import torch
from peft import LoraConfig, PeftModel, get_peft_model, prepare_model_for_kbit_training
from transformers import (AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig, DataCollatorForSeq2Seq,
                          TrainingArguments, set_seed)
from transformers.trainer_utils import get_last_checkpoint
from transformers.utils import logging as hf_logging

from spoilerqa.data import DEFAULT_DATA_DIR, TYPES, load_jsonl, load_split, post_text, spoiler_type, write_jsonl
from spoilerqa.metrics import as_text, bleu, ensure_nltk_data

hf_logging.set_verbosity_error()
ensure_nltk_data()
pd.set_option("display.max_colwidth", 100)

SMOKE = os.environ.get("SMOKE", "0") == "1"
DATA_DIR = Path(os.environ.get("DATA_DIR", DEFAULT_DATA_DIR))
WORK_DIR = Path("smoke") if SMOKE else Path(".")
RUNS_DIR, OUTPUT_DIR = WORK_DIR / "runs", WORK_DIR / "output"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
M2_DIR = Path("inputs/milestone2")  # frozen copies of the Milestone 2 outputs we reuse
SEED = 42

HAS_GPU = torch.cuda.is_available()
USE_BF16 = HAS_GPU and torch.cuda.is_bf16_supported()
DEVICE = "cuda" if HAS_GPU else "cpu"  # a laptop runs the smoke test on the CPU

# Smoke-test limits (None / -1 = use everything).
MAX_TRAIN = 16 if SMOKE else None
MAX_EVAL = 8 if SMOKE else None
MAX_STEPS = 2 if SMOKE else -1

from spoilerqa.gen import (GenerationEvalTrainer, causal_generate, llama_answer_prefix, llama_prompt_ids,
                           llama_target, parse_llama, spoiler_visible, to_prediction)

NAME = "llama-qlora"
MODEL = "HuggingFaceTB/SmolLM2-135M-Instruct" if SMOKE else "meta-llama/Llama-3.1-8B-Instruct"
USE_4BIT = HAS_GPU
LEARNING_RATES = [5e-5, 1e-4, 2e-4]
EPOCHS, GRAD_ACCUM = 2, (2 if SMOKE else 16)
ARTICLE_TOKENS = 256 if SMOKE else 1600   # the article is cut to this many tokens
MAX_ANSWER_TOKENS = 48 if SMOKE else 256  # training answers are cut to this many tokens
MAX_NEW_TOKENS = 32 if SMOKE else 256     # generation limit
GEN_BATCH = 8                             # posts per generation batch
LORA = dict(r=16, lora_alpha=32, lora_dropout=0.05, target_modules="all-linear", task_type="CAUSAL_LM")

print("GPU:", torch.cuda.get_device_name(0) if HAS_GPU else "none", "| bf16:", USE_BF16, "| smoke test:", SMOKE,
      "| model:", MODEL)

print("4-bit loading:", USE_4BIT)

GPU: NVIDIA L4 | bf16: True | smoke test: False | model: meta-llama/Llama-3.1-8B-Instruct
4-bit loading: True


Load the tokenizer. This is also the access check: if the Llama license hasn't been accepted or no token is set, this cell fails with Hugging Face's "gated repo" error. In that case do the two steps at the top and run again.

Llama has no padding token, so we use its reserved `<|finetune_right_pad_id|>` token. Padding never counts in the loss.

In [4]:
tokenizer = AutoTokenizer.from_pretrained(MODEL)
if tokenizer.pad_token is None or tokenizer.pad_token == tokenizer.eos_token:
    reserved = "<|finetune_right_pad_id|>"
    tokenizer.pad_token = reserved if reserved in tokenizer.get_vocab() else tokenizer.eos_token
print("eos:", tokenizer.eos_token, "| pad:", tokenizer.pad_token)

eos: <|eot_id|> | pad: <|finetune_right_pad_id|>


Load the splits. In the smoke test we keep only the first few posts of each.

In [5]:
train = load_split(DATA_DIR, "train")[:MAX_TRAIN]
dev = load_split(DATA_DIR, "dev")[:MAX_EVAL]
test = load_split(DATA_DIR, "test")[:MAX_EVAL]
print({"train": len(train), "dev": len(dev), "test": len(test)})

{'train': 3157, 'dev': 794, 'test': 1000}


Build the training examples. Each one is the chat prompt (system message with the task, then the post and the article cut to 1,600 tokens), followed by the answer and the end-of-turn token. The labels are `-100` (ignored) for every prompt token, so the model learns only to write answers. The cell prints the counts, the lengths and one full example.

In [6]:
def llama_features(records):
    features, hidden = [], 0
    for r in records:
        prompt, kept = llama_prompt_ids(tokenizer, r, ARTICLE_TOKENS)
        if not spoiler_visible(r, kept):
            hidden += 1
            continue
        answer = tokenizer(llama_target(r) + tokenizer.eos_token, add_special_tokens=False)["input_ids"]
        answer = answer[:MAX_ANSWER_TOKENS]
        features.append({"input_ids": prompt + answer, "attention_mask": [1] * (len(prompt) + len(answer)),
                         "labels": [-100] * len(prompt) + answer})
    return features, hidden


train_features, hidden = llama_features(train)
dev_features, _ = llama_features(dev)
lengths = pd.Series([len(f["input_ids"]) for f in train_features])
print(f"train: {len(train)} posts -> {len(train_features)} examples ({hidden} left out: spoiler after the cut)")
print(f"tokens per example: median {int(lengths.median())}, max {lengths.max()}, total {lengths.sum():,}")
print("\n--- example (prompt + answer) ---\n" + tokenizer.decode(train_features[0]["input_ids"])[-700:])

train: 3157 posts -> 3135 examples (22 left out: spoiler after the cut)
tokens per example: median 638, max 1999, total 2,428,525

--- example (prompt + answer) ---
y typically is all business during football season. Welker probably should have known what he was getting into when reaching out to his buddy.
"That’s the only thing we really have planned," Welker said of his upcoming workout with Brady. "It’s just funny. I’m sitting there trying to have dinner. ‘Hey, get your ass up here and let’s go throw.’ I’m like, ‘Aw jeez, man.’ He’s going to have me running like 2-minute drills in his backyard or something."
Maybe Brady will put a good word in for Welker down in Foxboro if the former Patriots wide receiver impresses him enough.<|eot_id|><|start_header_id|>assistant<|end_header_id|>

type: passage
spoiler: how about that morning we go throw?<|eot_id|>


**Loading the model and predicting.**
- `load_base_model` loads Llama in 4-bit NF4 with bf16 compute (or plainly, in the laptop smoke test).
- `llama_predict` builds the prompts, generates greedily in batches and parses each answer (`parse_llama`). If the answer has no valid `type:` line, the type is guessed from the spoiler.
- With `use_gold_types=True`, the answer is started for the model as `type: <gold type>\nspoiler:`, for the gold-type analysis.

In [7]:
def load_base_model():
    if USE_4BIT:
        quantization = BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_quant_type="nf4", bnb_4bit_use_double_quant=True,
                                          bnb_4bit_compute_dtype=torch.bfloat16)
        model = AutoModelForCausalLM.from_pretrained(MODEL, quantization_config=quantization, dtype=torch.bfloat16,
                                                     device_map={"": 0})
    else:
        model = AutoModelForCausalLM.from_pretrained(MODEL, dtype=torch.bfloat16 if USE_BF16 else torch.float32)
        model.to(DEVICE)
    model.config.pad_token_id = tokenizer.pad_token_id
    model.generation_config.pad_token_id = tokenizer.pad_token_id
    return model


def llama_predict(model, records, use_gold_types=False):
    prompts, prefixes = [], []
    for r in records:
        ids, _ = llama_prompt_ids(tokenizer, r, ARTICLE_TOKENS)
        prefix = llama_answer_prefix(spoiler_type(r)) if use_gold_types else ""
        if prefix:
            ids = ids + tokenizer(prefix, add_special_tokens=False)["input_ids"]
        prompts.append(ids)
        prefixes.append(prefix)
    texts = causal_generate(model, tokenizer, prompts, MAX_NEW_TOKENS, GEN_BATCH, bf16=USE_BF16)
    return [to_prediction(r["uuid"], *parse_llama(prefix + text)) for r, prefix, text in zip(records, prefixes, texts)]

One training run for one learning rate:
1. Load the 4-bit base model and prepare it for k-bit training (gradient checkpointing, and the few non-quantized layers kept in full precision).
2. Add fresh LoRA adapters and train for 2 epochs.
3. After each epoch, generate the dev spoilers and keep the epoch with the best dev BLEU.

Checkpoints hold the adapters and the optimizer state, so if the VM stops mid-run, running the notebook again resumes from the last saved epoch. The best adapters go to `runs/llama-qlora-lr<lr>/best`, the per-epoch scores to `run_info.json`.

In [8]:
def free_gpu():
    gc.collect()
    if HAS_GPU:
        torch.cuda.empty_cache()



def epoch_or_steps():
    # Evaluate and save after every epoch; the smoke test does it after its 2 steps instead.
    if MAX_STEPS < 0:
        return dict(eval_strategy="epoch", save_strategy="epoch")
    return dict(eval_strategy="steps", eval_steps=MAX_STEPS, save_strategy="steps", save_steps=MAX_STEPS)


def train_llama(learning_rate):
    run_dir = RUNS_DIR / f"{NAME}-lr{learning_rate:g}"
    if (run_dir / "run_info.json").exists():
        print("already trained, reusing", run_dir / "run_info.json")
        return json.loads((run_dir / "run_info.json").read_text())
    set_seed(SEED)
    model = load_base_model()
    if USE_4BIT:
        model = prepare_model_for_kbit_training(model, use_gradient_checkpointing=True,
                                                gradient_checkpointing_kwargs={"use_reentrant": False})
    model = get_peft_model(model, LoraConfig(**LORA))
    trainable, total = model.get_nb_trainable_parameters()
    print(f"trainable parameters: {trainable:,} of {total:,} ({trainable / total:.2%})")
    args = TrainingArguments(
        output_dir=str(run_dir), learning_rate=learning_rate, num_train_epochs=EPOCHS, max_steps=MAX_STEPS,
        per_device_train_batch_size=1, gradient_accumulation_steps=GRAD_ACCUM,
        weight_decay=0.0, warmup_steps=0.03, lr_scheduler_type="cosine",  # warmup_steps < 1 means a ratio
        optim="paged_adamw_8bit" if USE_4BIT else "adamw_torch",
        gradient_checkpointing=True, gradient_checkpointing_kwargs={"use_reentrant": False},
        **epoch_or_steps(), save_total_limit=2, save_only_model=False,
        load_best_model_at_end=True, metric_for_best_model="bleu", greater_is_better=True,
        logging_steps=10, bf16=USE_BF16, use_cpu=not HAS_GPU, report_to="none", seed=SEED,
    )
    trainer = GenerationEvalTrainer(
        model=model, args=args, train_dataset=train_features, eval_dataset=dev_features, processing_class=tokenizer,
        data_collator=DataCollatorForSeq2Seq(tokenizer, padding=True, label_pad_token_id=-100),
        predict_fn=llama_predict, dev_records=dev,
    )
    last = get_last_checkpoint(str(run_dir)) if run_dir.exists() else None
    if last:
        print("resuming from", last)
    trainer.train(resume_from_checkpoint=last)
    trainer.save_model(str(run_dir / "best"))
    history = trainer.state.log_history
    per_epoch = [h["eval_bleu"] for h in history if "eval_bleu" in h]
    info = {"learning_rate": learning_rate, "dev_bleu": max(per_epoch), "dev_bleu_per_epoch": per_epoch,
            "dev_type_balanced_accuracy_per_epoch": [h["eval_type_balanced_accuracy"] for h in history
                                                     if "eval_type_balanced_accuracy" in h],
            "train_minutes": round(next(h["train_runtime"] for h in history if "train_runtime" in h) / 60, 1),
            "trainable_parameters": trainable, "run_dir": str(run_dir)}
    (run_dir / "run_info.json").write_text(json.dumps(info, indent=2) + "\n")
    del trainer, model
    free_gpu()
    return info

**The learning-rate sweep.** Dev BLEU and type balanced accuracy after every epoch, saved to `output/llama-qlora-sweep.json`.

In [9]:
sweep = []
for lr in LEARNING_RATES:
    print(f"=== learning rate {lr:g} ===")
    sweep.append(train_llama(lr))

(OUTPUT_DIR / f"{NAME}-sweep.json").write_text(json.dumps(sweep, indent=2) + "\n")
pd.DataFrame(sweep)[["learning_rate", "dev_bleu", "dev_bleu_per_epoch", "dev_type_balanced_accuracy_per_epoch",
                     "train_minutes"]]

=== learning rate 5e-05 ===


Loading weights: 100%|██████████| 291/291 [00:59<00:00,  4.88it/s]


trainable parameters: 41,943,040 of 8,072,204,288 (0.52%)


/home/dhanush/project/.venv/lib/python3.12/site-packages/triton/knobs.py:504: UserWarning: TRITON_DEFAULT_FP_FUSION is unset: defaulting to FP fusion ENABLED for backward compatibility in Triton 3.9. The default will change to DISABLED in Triton 3.10; set TRITON_DEFAULT_FP_FUSION=1 to keep fusion enabled or =0 to adopt the new default early and silence this warning.
  warn_once(FP_FUSION_DEFAULT_WARNING)


{'loss': '0.5983', 'grad_norm': '1.904', 'learning_rate': '3.75e-05', 'epoch': '0.05104'}
{'loss': '0.2505', 'grad_norm': '0.8353', 'learning_rate': '4.996e-05', 'epoch': '0.1021'}
{'loss': '0.2892', 'grad_norm': '0.7758', 'learning_rate': '4.975e-05', 'epoch': '0.1531'}
{'loss': '0.248', 'grad_norm': '0.7081', 'learning_rate': '4.938e-05', 'epoch': '0.2041'}
{'loss': '0.2075', 'grad_norm': '1.016', 'learning_rate': '4.884e-05', 'epoch': '0.2552'}
{'loss': '0.23', 'grad_norm': '0.9203', 'learning_rate': '4.814e-05', 'epoch': '0.3062'}
{'loss': '0.1871', 'grad_norm': '0.7537', 'learning_rate': '4.728e-05', 'epoch': '0.3573'}
{'loss': '0.1632', 'grad_norm': '0.7', 'learning_rate': '4.626e-05', 'epoch': '0.4083'}
{'loss': '0.1895', 'grad_norm': '0.5818', 'learning_rate': '4.51e-05', 'epoch': '0.4593'}
{'loss': '0.1782', 'grad_norm': '0.7797', 'learning_rate': '4.381e-05', 'epoch': '0.5104'}
{'loss': '0.1747', 'grad_norm': '0.9765', 'learning_rate': '4.238e-05', 'epoch': '0.5614'}
{'loss':

Loading weights: 100%|██████████| 291/291 [00:04<00:00, 70.54it/s]


trainable parameters: 41,943,040 of 8,072,204,288 (0.52%)
{'loss': '0.5438', 'grad_norm': '2.04', 'learning_rate': '7.5e-05', 'epoch': '0.05104'}
{'loss': '0.2403', 'grad_norm': '0.797', 'learning_rate': '9.992e-05', 'epoch': '0.1021'}
{'loss': '0.2818', 'grad_norm': '0.6548', 'learning_rate': '9.951e-05', 'epoch': '0.1531'}
{'loss': '0.2365', 'grad_norm': '0.6011', 'learning_rate': '9.876e-05', 'epoch': '0.2041'}
{'loss': '0.2024', 'grad_norm': '0.8371', 'learning_rate': '9.768e-05', 'epoch': '0.2552'}
{'loss': '0.2233', 'grad_norm': '0.8116', 'learning_rate': '9.627e-05', 'epoch': '0.3062'}
{'loss': '0.1776', 'grad_norm': '0.6547', 'learning_rate': '9.455e-05', 'epoch': '0.3573'}
{'loss': '0.1577', 'grad_norm': '0.671', 'learning_rate': '9.252e-05', 'epoch': '0.4083'}
{'loss': '0.1848', 'grad_norm': '0.5198', 'learning_rate': '9.021e-05', 'epoch': '0.4593'}
{'loss': '0.1702', 'grad_norm': '0.59', 'learning_rate': '8.761e-05', 'epoch': '0.5104'}
{'loss': '0.1714', 'grad_norm': '0.8226

Loading weights: 100%|██████████| 291/291 [00:03<00:00, 73.77it/s]


trainable parameters: 41,943,040 of 8,072,204,288 (0.52%)
{'loss': '0.498', 'grad_norm': '1.186', 'learning_rate': '0.00015', 'epoch': '0.05104'}
{'loss': '0.222', 'grad_norm': '0.7037', 'learning_rate': '0.0001998', 'epoch': '0.1021'}
{'loss': '0.2721', 'grad_norm': '0.6511', 'learning_rate': '0.000199', 'epoch': '0.1531'}
{'loss': '0.2192', 'grad_norm': '0.6031', 'learning_rate': '0.0001975', 'epoch': '0.2041'}
{'loss': '0.2075', 'grad_norm': '0.9643', 'learning_rate': '0.0001954', 'epoch': '0.2552'}
{'loss': '0.2228', 'grad_norm': '0.895', 'learning_rate': '0.0001925', 'epoch': '0.3062'}
{'loss': '0.1712', 'grad_norm': '0.6693', 'learning_rate': '0.0001891', 'epoch': '0.3573'}
{'loss': '0.1562', 'grad_norm': '0.4299', 'learning_rate': '0.000185', 'epoch': '0.4083'}
{'loss': '0.1816', 'grad_norm': '0.6015', 'learning_rate': '0.0001804', 'epoch': '0.4593'}
{'loss': '0.1679', 'grad_norm': '0.551', 'learning_rate': '0.0001752', 'epoch': '0.5104'}
{'loss': '0.1769', 'grad_norm': '0.6492'

,learning_rate,dev_bleu,dev_bleu_per_epoch,dev_type_balanced_accuracy_per_epoch,train_minutes
0,0.00005,0.490474,"[0.475482, 0.490474]","[0.729448, 0.755385]",241.5
1,0.00010,0.501813,"[0.46586, 0.501813]","[0.740327, 0.773125]",239.2
2,0.00020,0.503811,"[0.482422, 0.503811]","[0.759157, 0.773671]",242.7


## Prediction

Load the base model with the best run's adapters and generate dev and test, plus test with the gold types given (analysis only). Files: `llama-qlora-{dev,test}.jsonl` and `llama-qlora-goldtypes-test.jsonl`.

In [10]:
best_run = max(sweep, key=lambda info: info["dev_bleu"])
print("best learning rate:", best_run["learning_rate"], "| dev BLEU:", round(best_run["dev_bleu"], 4))
model = PeftModel.from_pretrained(load_base_model(), Path(best_run["run_dir"]) / "best").eval()

write_jsonl(llama_predict(model, dev), OUTPUT_DIR / f"{NAME}-dev.jsonl")
write_jsonl(llama_predict(model, test), OUTPUT_DIR / f"{NAME}-test.jsonl")
write_jsonl(llama_predict(model, test, use_gold_types=True), OUTPUT_DIR / f"{NAME}-goldtypes-test.jsonl")
(OUTPUT_DIR / f"{NAME}-settings.json").write_text(json.dumps(
    {"model": MODEL, "learning_rate": best_run["learning_rate"], "lora": LORA, "quantization": "nf4" if USE_4BIT else None,
     "article_tokens": ARTICLE_TOKENS, "max_new_tokens": MAX_NEW_TOKENS, "decoding": "greedy"}, indent=2) + "\n")
del model
free_gpu()
print(sorted(p.name for p in OUTPUT_DIR.glob(f"{NAME}*")))

best learning rate: 0.0002 | dev BLEU: 0.5038


Loading weights: 100%|██████████| 291/291 [00:03<00:00, 73.79it/s]


['llama-qlora-dev.jsonl', 'llama-qlora-goldtypes-test.jsonl', 'llama-qlora-settings.json', 'llama-qlora-sweep.json', 'llama-qlora-test.jsonl']


## Scores

`score.py` (the Milestone 2 evaluation script, copied here) scores each prediction file exactly as from the command line: BLEU (the main number), METEOR, BERTScore, exact match and token F1, overall and per gold type, plus type balanced accuracy. The JSON files go to `output/scores/`. The smoke test skips BERTScore and scores only the posts it predicted.

In [11]:
def gold_file(split, records):
    if not SMOKE:
        return DATA_DIR / f"{split}.jsonl"
    path = WORK_DIR / f"gold-{split}.jsonl"  # the smoke test predicts only a few posts
    write_jsonl(records, path)
    return path


def run_score(split, records, name):
    cmd = [sys.executable, "score.py", "--gold", str(gold_file(split, records)),
           "--pred", str(OUTPUT_DIR / f"{name}-{split}.jsonl"),
           "--output", str(OUTPUT_DIR / "scores" / f"{name}-{split}.json")]
    if SMOKE:
        cmd.append("--no-bertscore")
    print(f"$ python score.py --gold {cmd[3]} --pred {cmd[5]}")
    print(subprocess.run(cmd, capture_output=True, text=True, check=True).stdout)


run_score("dev", dev, NAME)
run_score("test", test, NAME)
run_score("test", test, f"{NAME}-goldtypes")

$ python score.py --gold /home/dhanush/project/data/splits/dev.jsonl --pred output/llama-qlora-dev.jsonl
Scored 794 posts (0 missing predictions)

spoiler              all    phrase   passage     multi
n                    794       333       318       143
bleu              0.5032    0.6495    0.3734    0.4514
meteor            0.5424    0.4029    0.4830    0.6718
bertscore         0.9321    0.9525    0.9149    0.9231
exact-match       0.3678    0.5886    0.1855    0.2587
token-f1          0.6164    0.7261    0.5020    0.6154

spoiler type: balanced accuracy 0.7769
  phrase   precision 0.7971  recall 0.8138  f1 0.8053
  passage  precision 0.7879  recall 0.8176  f1 0.8025
  multi    precision 0.8065  recall 0.6993  f1 0.7491

BLEU (all): 0.5032

$ python score.py --gold /home/dhanush/project/data/splits/test.jsonl --pred output/llama-qlora-test.jsonl
Scored 1000 posts (0 missing predictions)

spoiler              all    phrase   passage     multi
n                   1000       423      

## A few test predictions

Six random test posts with their gold and predicted spoilers, to sanity-check the numbers above.

In [12]:
test_preds = {p["uuid"]: p for p in load_jsonl(OUTPUT_DIR / f"{NAME}-test.jsonl")}
sample = pd.DataFrame([{
    "post": post_text(r), "gold type": spoiler_type(r), "predicted type": test_preds[r["uuid"]]["spoilerType"],
    "gold spoiler": as_text(r["spoiler"]), "predicted spoiler": as_text(test_preds[r["uuid"]]["spoiler"]),
} for r in test])
sample.sample(min(6, len(sample)), random_state=0)

,post,gold type,predicted type,gold spoiler,predicted spoiler
993,Inside our three-month effort to attend Apple's iPhone 7 launch party,passage,passage,"The truth though is that large tech companies, especially in Silicon Valley, often use access to...",I have placed you on the waiting list as all media seats are allocated. We expect high take up f...
859,Mum takes 'funny' photo of toddler - breaks down over chilling truth behind it,passage,passage,"""She was practising for a lockdown drill at her preschool and what you should do if you are stuc...",She was practising for a lockdown drill at her preschool and what you should do if you are stuck...
298,Duterte plans to ‘SHOOT and KILL’ these people; find out who they are,passage,phrase,business owners that he will kill them if they continue to practice endo or contractualization,"business owners who practice contractualization or ""endo"""
553,LOOK: This Is Definitely The Biggest Spring 2014 Trend,passage,phrase,popping Spring colors and mixed prints,Bright Colors and Mixed Prints
672,Lot of Pokémon Go players demanding refunds after latest update due to this reason,passage,passage,"game would no longer be supported in unlocked, rooted or jail-broken devices","A warning that the game would no longer be supported in unlocked, rooted or jail-broken devices ..."
971,"Former ""SNL"" star heads to ""House of Lies""",phrase,phrase,Fred Armisen,Fred Armisen
